In [ ]:
import os
from pathlib import Path
DATA = Path(os.environ.get('ENSO_DATA_ROOT', Path.cwd() / 'data'))

# 读数据
data_path_aprc = str(DATA / "aprcandstd.mat")
lon_aprc = sio.loadmat(data_path_aprc)['lonaprc'].flatten()
lat = sio.loadmat(data_path_aprc)['lat'].flatten()
idx_lat = np.where((lat >= -30) & (lat <= 30))[0]
idx_lon = np.where((lon_aprc >= 40) & (lon_aprc <= 300))[0]
apprecession = sio.loadmat(data_path_aprc)['apprecession']
anomalystdprecesion = sio.loadmat(data_path_aprc)['anomalystdprecesion']
X, Y = np.meshgrid(lon_aprc[idx_lon]-180, lat[idx_lat])

ap_105 = apprecession[idx_lon][:, idx_lat, 6].T
threecell_105 = anomalystdprecesion[idx_lon][:, idx_lat, 6].T
ap_285 = apprecession[idx_lon][:, idx_lat, 18].T
threecell_285 = anomalystdprecesion[idx_lon][:, idx_lat, 18].T

In [ ]:
# 数据加载
data_path_wind = str(DATA / "windnew.mat")
lon_wind = sio.loadmat(data_path_wind)['lon_wind'].flatten()
lat_wind = sio.loadmat(data_path_wind)['lat_wind'].flatten()
ano_u = sio.loadmat(data_path_wind)['ano_u'].T
h = sio.loadmat(data_path_wind)['h'].T
lonh, lath = np.where(h.T == 1)
aannual = lat_wind [lath]
bannual = lon_wind[lonh] 

In [ ]:
import itertools
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
import scipy.io as sio
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind  # 双样本t检验
import cartopy.crs as ccrs  # 地图
import cartopy.feature as cfeature
import colormaps as cmaps  # 确保 colormaps 库已安装

# 统一全局字体设置
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 11,
    'font.weight': 'normal',
    'axes.linewidth': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in'
})



# colorbar
colorcode4 = np.loadtxt(str(DATA / "colormaps" / "sunshine_9lev.rgb"))
colorcode4 = colorcode4 / 256
custom_cmap4 = ListedColormap(colorcode4)

# 绘图
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1], hspace=0.25)

# 子图1
ax1 = fig.add_subplot(gs[0], projection=ccrs.PlateCarree(central_longitude=180))
ax1.set_extent([40, 300, -30, 30], crs=ccrs.PlateCarree())
ax1.add_feature(cfeature.COASTLINE, edgecolor='0.7', linewidth=1.5)

colorcode = np.loadtxt(str(DATA / "colormaps" / "WhiteYellowOrangeRed (3).rgb"))
colorcode = colorcode / 256
custom_cmap = ListedColormap(colorcode)

# 填充等高线
c1 = ax1.contourf(X, Y, threecell_105, levels=np.arange(0.8, 1.6, 0.001), cmap=custom_cmap, extend='both')
h1 = ax1.contour(X, Y, ap_105, levels=np.arange(5, 6, 1), colors='b', linestyles='-', linewidths=2)
ax1.clabel(h1, inline=True, fontsize=11, fmt='5 mm/day', colors='b')
# 设置坐标轴标签和刻度
ax1.set_xlim([-140, 120])
ax1.set_xticks(np.arange(-140, 140, 20))
ax1.set_xticklabels(['40°E', '60°E', '80°E', '100°E', '120°E', '140°E', '160°E', '180°', '160°W', '140°W', '120°W', '100°W', '80°W', '60°W'], fontsize=11, fontweight='normal')
ax1.tick_params(axis='x', labelbottom=False)  # 隐藏x轴刻度标签
ax1.set_ylim([-30, 30])
ax1.set_yticks(np.arange(-30, 40, 15))
ax1.set_yticklabels(['30°S', '15°S', 'EQ', '15°N', '30°N'], fontsize=11, fontweight='normal')
# ax1.grid(True, linestyle='-.', color='lightgray')

# 添加标题
ax1.text(-140, 35, '(a) Precessional Phase 90°', color='black', fontsize=11, fontweight='normal')

# 子图2
ax2 = fig.add_subplot(gs[1], projection=ccrs.PlateCarree(central_longitude=180))
ax2.set_extent([40, 300, -30, 30], crs=ccrs.PlateCarree())
ax2.add_feature(cfeature.COASTLINE, edgecolor='0.7', linewidth=1.5)

# 填充等高线
c2 = ax2.contourf(X, Y, threecell_285, levels=np.arange(0.8, 1.6, 0.01), cmap=custom_cmap, extend='both')
h2 = ax2.contour(X, Y, ap_285, levels=np.arange(5, 6, 1), colors='r', linestyles='-', linewidths=2)
ax2.clabel(h2, inline=True, fontsize=11, fmt='5 mm/day', colors='r')
# 设置坐标轴标签和刻度
ax2.set_xlim([-140, 120])
ax2.set_xticks(np.arange(-140, 140, 20))
ax2.set_xticklabels(['40°E', '60°E', '80°E', '100°E', '120°E', '140°E', '160°E', '180°', '160°W', '140°W', '120°W', '100°W', '80°W', '60°W'], fontsize=11, fontweight='normal')
ax2.tick_params(axis='x', labelbottom=False)  # 隐藏x轴刻度标签
ax2.set_ylim([-30, 30])
ax2.set_yticks(np.arange(-30, 40, 15))
ax2.set_yticklabels(['30°S', '15°S', 'EQ', '15°N', '30°N'], fontsize=11, fontweight='normal')
# ax2.grid(True, linestyle='-.', color='lightgray')

# 添加标题
ax2.text(-140, 35, '(b) Precessional Phase 270°', color='black', fontsize=11, fontweight='normal')

# 子图3
ax3 = fig.add_subplot(gs[2], projection=ccrs.PlateCarree(central_longitude=180))
ax3.set_extent([40, 300, -30, 30], crs=ccrs.PlateCarree())
ax3.add_feature(cfeature.COASTLINE, edgecolor='0.7', linewidth=1.5)

# 填充等高线
h3 = ax3.contour(X, Y, ap_105, levels=np.arange(5, 6, 1), colors='b', linestyles='-', linewidths=2)
h4 = ax3.contour(X, Y, ap_285, levels=np.arange(5, 6, 1), colors='r', linestyles='-', linewidths=2)



# 创建网格
X2, Y2 = np.meshgrid(lon_wind, lat_wind)
cwind = ax3.contourf(X2-180, Y2, ano_u, levels=np.arange(-4,4, 0.1), cmap=custom_cmap1, extend='both')
ax3.scatter(bannual-180, aannual, color='lightgray', s=0.2, marker='p', label='显著性点95%', alpha=0.7, zorder=2)



# 设置坐标轴标签和刻度
ax3.set_xlim([-140, 120])
ax3.set_xticks(np.arange(-140, 140, 20))
ax3.set_xticklabels(['40°E', '60°E', '80°E', '100°E', '120°E', '140°E', '160°E', '180°', '160°W', '140°W', '120°W', '100°W', '80°W', '60°W'], rotation=0, fontsize=11, fontweight='normal')
ax3.set_ylim([-30, 30])
ax3.set_yticks(np.arange(-30, 40, 15))
ax3.set_yticklabels(['30°S', '15°S', 'EQ', '15°N', '30°N'], fontsize=11, fontweight='normal')
# ax3.grid(True, linestyle='-.', color='lightgray')

# 添加标题
ax3.text(-140, 35, '(c) Anomaly', color='black', fontsize=11, fontweight='normal')


# 添加垂直 colorbar，位于子图 (a) 和 (b) 右侧
cax1 = fig.add_axes([0.86, 0.45, 0.015, 0.4])  # [left, bottom, width, height]，垂直布局
colorbar1 = fig.colorbar(c1, cax=cax1, orientation='vertical')
colorbar1.set_label('SSTA Std Dev [C°]', fontsize=11, fontweight='normal')
ticks = np.arange(0.8, 1.8, 0.2)
colorbar1.set_ticks(ticks)
ticklabels = ['%.1f' % tick for tick in ticks]
colorbar1.set_ticklabels(ticklabels, fontsize=11, fontweight='normal')



# 添加垂直 colorbar，位于子图 (a) 和 (b) 右侧
cax2 = fig.add_axes([0.86, 0.12, 0.015, 0.2])  # [left, bottom, width, height]，垂直布局
colorbar1 = fig.colorbar(cwind, cax=cax2, orientation='vertical')
colorbar1.set_label('Zonal wind anomaly [m/s]', fontsize=11, fontweight='normal')
ticks = np.arange(-4, 4.01, 2)
colorbar1.set_ticks(ticks)
# 自定义刻度标签：0 显示为 "0"，其他保持 %.1f 格式
ticklabels = ['0' if tick == 0 else '%.1f' % tick for tick in ticks]
colorbar1.set_ticklabels(ticklabels, fontsize=11, fontweight='normal')
# 保存
plt.savefig('S4.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
# 找出数组的最大值和最小值
max_value = np.max( threecell_105)
min_value = np.min( threecell_105)
print("最大值:", max_value)
print("最小值:", min_value)
max_value = np.max( diff)
min_value = np.min(diff)
print("diff最大值:", max_value)
print("diff最小值:", min_value)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

# 读取.rgb文件中的颜色
colorcode_path = str(DATA / "colormaps" / "MPL_PRGn.rgb")

# 读取RGB颜色
colorcode = np.loadtxt(colorcode_path)

# 确保读取到的颜色数据是正确的，检查文件的形状
print(colorcode.shape)

# 创建自定义颜色映射
custom_cmap1 = ListedColormap(colorcode / 1)  # 将RGB值缩放到0-1范围内

# 测试色带
data = np.random.rand(8, 8)  # 创建8x8的随机数据
plt.imshow(data, cmap=custom_cmap1)
plt.colorbar()  # 添加颜色条
plt.show()